# Модельні дослідження: Високоточне машинне навчання на основі Extreme Gradient Boosting (XGBoost)

* **Тема:** «Інтелектуальна система оцінки ризиків захворювань на основі методів пояснюваного штучного інтелекту»
* **Спеціальність:** 122 «Комп’ютерні науки»
* **Метод моделювання:** **XGBoost (Extreme Gradient Boosting)** — високоточний ансамблевий алгоритм градієнтного бустингу над вирішальними деревами, оптимізований через розклад втрат другого порядку Тейлора та L1/L2 регуляризацію. Забезпечує нативну швидкісну сумісність з алгоритмом TreeSHAP.


In [1]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Бібліотеки успішно імпортовано!")


### 1. Завантаження підготовлених клінічних вибірок даних (253,680 пацієнтів)

In [2]:
data_dir = os.path.join('..', 'Datasets', 'processed')
X_train = pd.read_csv(os.path.join(data_dir, 'X_train.csv'))
y_train = pd.read_csv(os.path.join(data_dir, 'y_train.csv')).values.ravel()
X_test = pd.read_csv(os.path.join(data_dir, 'X_test.csv'))
y_test = pd.read_csv(os.path.join(data_dir, 'y_test.csv')).values.ravel()

print(f"Тренувальна вибірка: {X_train.shape[0]:,} пацієнтів, {X_train.shape[1]} клінічних факторів")
print(f"Тестова вибірка:     {X_test.shape[0]:,} пацієнтів")
print(f"Частка позитивного класу (серцево-судинні події): {np.mean(y_train):.2%}")


### 2. Навчання та оптимізація моделі XGBoost
Модель навчається на повному масиві 202,944 пацієнтів із захистом від перенавчання (L1/L2 регуляризація, subsample, глибина 5).

In [3]:
model = XGBClassifier(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.5,
    reg_lambda=2.0,
    min_child_weight=3,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

t0 = time.time()
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)
print(f"Навчання завершено за {time.time() - t0:.2f} сек.")


### 3. Комплексна клінічна оцінка якості моделі

In [4]:
y_probs = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_probs)
pr_auc = average_precision_score(y_test, y_probs)

fpr, tpr, thresholds = roc_curve(y_test, y_probs)
youden_idx = np.argmax(tpr - fpr)
optimal_threshold = float(thresholds[youden_idx])

y_pred_std = (y_probs >= 0.50).astype(int)
y_pred_opt = (y_probs >= optimal_threshold).astype(int)

print(f"ROC-AUC: {roc_auc:.4f} ({roc_auc*100:.2f}%)")
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"Загальна точність (Accuracy при порозі 0.5): {accuracy_score(y_test, y_pred_std):.2%}")
print(f"Збалансована точність (Balanced Accuracy при порозі {optimal_threshold:.2f}): {balanced_accuracy_score(y_test, y_pred_opt):.2%}")


### 4. Візуалізація кривих ROC та Precision-Recall

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1.plot(fpr, tpr, color='#1E40AF', lw=2.5, label=f'XGBoost (AUC = {roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], color='#9CA3AF', linestyle='--', lw=1.5, label='Випадковий класифікатор (0.50)')
ax1.scatter(fpr[youden_idx], tpr[youden_idx], color='#DC2626', s=80, zorder=5, label=f'Оптимальний поріг ({optimal_threshold:.2f})')
ax1.set_title('Крива ROC (Дискримінаційна здатність)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Хибнопозитивна частка (1 - Специфічність)')
ax1.set_ylabel('Істиннопозитивна частка (Чутливість)')
ax1.legend(loc='lower right')

# PR Curve
prec, rec, _ = precision_recall_curve(y_test, y_probs)
ax2.plot(rec, prec, color='#059669', lw=2.5, label=f'XGBoost (PR-AUC = {pr_auc:.3f})')
ax2.axhline(y=np.mean(y_test), color='#9CA3AF', linestyle='--', lw=1.5, label=f'Базовий рівень ({np.mean(y_test):.1%})')
ax2.set_title('Крива Precision-Recall (Якість для рідкісного класу)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Повнота (Recall)')
ax2.set_ylabel('Точність (Precision)')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


### 5. Збереження моделі та метрик для подальшого XAI аналізу

In [6]:
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)

joblib.dump(model, os.path.join(models_dir, 'best_heart_disease_xgb.joblib'))
model.save_model(os.path.join(models_dir, 'best_heart_disease_xgb.json'))
print("Модель успішно збережена для веб-застосунку та аналізу SHAP!")
